In [17]:
# phase2_MMI_sweep.py
# MEEP 3D FDTD: sweep MMI length for 50/50 splitting at 1550 nm
# Reference geometry: [R6], waveguide from Phase 1

import meep as mp
import numpy as np
import scipy.io
import matplotlib.pyplot as plt

In [27]:

# ── Units: µm, c = 1 ───────────────────────────────────────────────────────
lam = 1.55
f0  = 1.0 / lam
df  = 0.15 / lam   # source bandwidth

# ── Materials ───────────────────────────────────────────────────────────────
LNO  = mp.Medium(epsilon_diag=mp.Vector3(2.138**2, 2.211**2, 2.211**2))
SiO2 = mp.Medium(index=1.444)


# ── Material indices at 1550 nm (Zelmon et al. 1997 [R5]) ──────────────────
n_e  = 2.138   # LNO extraordinary (TE mode, X-cut: c-axis along x)
n_o  = 2.211   # LNO ordinary
n_ox = 1.444   # SiO₂ BOX
n_air = 1.0

In [19]:

# ── Fixed geometry parameters ───────────────────────────────────────────────
h       = 0.30    # LNO core height
wg_w    = 0.90    # input/output waveguide width
W_MMI   = 5.5     # MMI body width (sweep W in a second pass if needed)
taper_L = 2.0     # linear taper length (wg_w → W_MMI at port)
port_off = W_MMI / 4  # = 1.375 µm — paired interference offset from center [R6]
pml_d   = 1.5     # PML thickness (µm)
wg_len  = 3.0     # input/output waveguide stub length beyond taper


In [20]:

def build_geometry(L_mmi, sx, sy, sz):
    """Build MEEP geometry list for 2×2 MMI with given body length."""
    x_mmi_c = 0.0    # MMI center at x = 0
    half_L  = L_mmi / 2

    geom = []

    # SiO₂ BOX (below LNO)
    geom.append(mp.Block(
        size=mp.Vector3(sx, sy, 2.0),
        center=mp.Vector3(0, 0, -(h/2 + 1.0)),
        material=SiO2
    ))

    # ── Input side (x < -half_L) ─────────────────────────────────────────
    # Upper input waveguide stub + taper
    x_in_wg_start = -(half_L + taper_L + wg_len)
    # Waveguide stub
    geom.append(mp.Block(
        size=mp.Vector3(wg_len, wg_w, h),
        center=mp.Vector3(x_in_wg_start + wg_len/2, +port_off, 0),
        material=LNO
    ))
    # Taper (approximated as block at full width — for accuracy use TrapezoidBlock
    # or break into small steps; sufficient for sweep at res=20)
    geom.append(mp.Block(
        size=mp.Vector3(taper_L, (wg_w + W_MMI/2)/2, h),
        center=mp.Vector3(-half_L - taper_L/2, +port_off/2, 0),
        material=LNO
    ))

    # Lower input waveguide (unused input port — terminated, or add second source)
    geom.append(mp.Block(
        size=mp.Vector3(wg_len, wg_w, h),
        center=mp.Vector3(x_in_wg_start + wg_len/2, -port_off, 0),
        material=LNO
    ))

    # ── MMI body ─────────────────────────────────────────────────────────
    geom.append(mp.Block(
        size=mp.Vector3(L_mmi, W_MMI, h),
        center=mp.Vector3(0, 0, 0),
        material=LNO
    ))

    # ── Output side (x > +half_L) ─────────────────────────────────────────
    for sign in [+1, -1]:
        # Taper
        geom.append(mp.Block(
            size=mp.Vector3(taper_L, (wg_w + W_MMI/2)/2, h),
            center=mp.Vector3(+half_L + taper_L/2, sign * port_off/2, 0),
            material=LNO
        ))
        # Output waveguide stub
        geom.append(mp.Block(
            size=mp.Vector3(wg_len, wg_w, h),
            center=mp.Vector3(+half_L + taper_L + wg_len/2, sign * port_off, 0),
            material=LNO
        ))

    return geom



In [21]:
###################### 3D #################################
# def run_mmi_sim(L_mmi, resolution=20):
#     """
#     Run MEEP 3D simulation of 2×2 MMI.
#     Returns (T_out_upper, T_out_lower) normalized by straight-wg reference.
#     """
#     total_x = pml_d + wg_len + taper_L + L_mmi + taper_L + wg_len + pml_d
#     sy_cell  = W_MMI + 4.0 + 2 * pml_d
#     sz_cell  = h + 4.5 + 2 * pml_d

#     cell = mp.Vector3(total_x, sy_cell, sz_cell)
#     half_tx = total_x / 2

#     geom = build_geometry(L_mmi, total_x, sy_cell, sz_cell)

#     # Source: TE eigenmode at upper input waveguide
#     x_src = -half_tx + pml_d + 0.5
#     src = [mp.EigenModeSource(
#         src=mp.GaussianSource(f0, fwidth=df),
#         center=mp.Vector3(x_src, +port_off, 0),
#         size=mp.Vector3(0, wg_w + 1.5, h + 1.5),
#         eig_band=1,
#         eig_parity=mp.ODD_Y + mp.EVEN_Z,  # TE-like
#         direction=mp.X,
#     )]

#     # Output flux monitors at both output ports
#     x_mon = +half_tx - pml_d - 0.5
#     L_half = L_mmi / 2
#     mon_upper = mp.FluxRegion(
#         center=mp.Vector3(x_mon, +port_off, 0),
#         size=mp.Vector3(0, wg_w + 1.0, h + 1.0)
#     )
#     mon_lower = mp.FluxRegion(
#         center=mp.Vector3(x_mon, -port_off, 0),
#         size=mp.Vector3(0, wg_w + 1.0, h + 1.0)
#     )

#     sim = mp.Simulation(
#         cell_size=cell,
#         boundary_layers=[mp.PML(pml_d)],
#         geometry=geom,
#         sources=src,
#         resolution=resolution,
#         default_material=mp.Medium(index=1.0),
#     )

#     flux_upper = sim.add_flux(f0, df, 100, mon_upper)
#     flux_lower = sim.add_flux(f0, df, 100, mon_lower)

#     sim.run(until_after_sources=mp.stop_when_fields_decayed(
#         50, mp.Ex,
#         mp.Vector3(x_mon, +port_off, 0),
#         1e-4
#     ))

#     T_upper = mp.get_fluxes(flux_upper)[0]
#     T_lower = mp.get_fluxes(flux_lower)[0]

#     sim.reset_meep()
#     return T_upper, T_lower



In [22]:
###################### 2D #################################
def run_mmi_sim(L_mmi, resolution=10):
    total_x = pml_d + wg_len + taper_L + L_mmi + taper_L + wg_len + pml_d
    sy_cell  = W_MMI + 4.0 + 2 * pml_d   # only x and y in 2D

    cell = mp.Vector3(total_x, sy_cell)   # no z component → 2D
    half_tx = total_x / 2

    geom = []
    # SiO2 BOX — in 2D just set background index; or use a block
    # LNO MMI body
    geom.append(mp.Block(
        size=mp.Vector3(L_mmi, W_MMI),
        center=mp.Vector3(0, 0),
        material=LNO,
    ))
    # input/output waveguide stubs
    for sign in [+1, -1]:
        geom.append(mp.Block(
            size=mp.Vector3(wg_len + taper_L, wg_w),
            center=mp.Vector3(-(half_tx - pml_d - (wg_len+taper_L)/2), sign*port_off),
            material=LNO,
        ))
        geom.append(mp.Block(
            size=mp.Vector3(wg_len + taper_L, wg_w),
            center=mp.Vector3(+(half_tx - pml_d - (wg_len+taper_L)/2), sign*port_off),
            material=LNO,
        ))

    x_src = -half_tx + pml_d + 0.5
    src = [mp.EigenModeSource(
        src=mp.GaussianSource(f0, fwidth=df),
        center=mp.Vector3(x_src, +port_off),
        size=mp.Vector3(0, wg_w + 1.5),
        eig_band=1,
        eig_parity=mp.ODD_Y,
        direction=mp.X,
    )]

    x_mon = +half_tx - pml_d - 0.5
    sim = mp.Simulation(
        cell_size=cell,
        dimensions=2,                        # ← 2D
        boundary_layers=[mp.PML(pml_d)],
        geometry=geom,
        sources=src,
        resolution=resolution,
        default_material=mp.Medium(index=n_ox),  # background = SiO2 (substrate)
    )

    flux_upper = sim.add_flux(f0, df, 100,
        mp.FluxRegion(center=mp.Vector3(x_mon, +port_off),
                      size=mp.Vector3(0, wg_w + 1.0)))
    flux_lower = sim.add_flux(f0, df, 100,
        mp.FluxRegion(center=mp.Vector3(x_mon, -port_off),
                      size=mp.Vector3(0, wg_w + 1.0)))

    sim.run(until_after_sources=mp.stop_when_fields_decayed(
        50, mp.Ez,                           # Ez in 2D TE
        mp.Vector3(x_mon, +port_off), 1e-4))

    T_upper = mp.get_fluxes(flux_upper)[0]
    T_lower = mp.get_fluxes(flux_lower)[0]
    sim.reset_meep()
    return T_upper, T_lower

In [23]:
# # ── Reference simulation (straight waveguide, for normalization) ──────────
# ###################### 3D #################################

# def run_reference(resolution=20):
#     """Straight waveguide — measures input power P_ref."""
#     ref_len = 20.0   # arbitrary straight length
#     total_x = pml_d + wg_len + ref_len + wg_len + pml_d
#     sy_c = wg_w + 3.0 + 2 * pml_d
#     sz_c = h + 4.0 + 2 * pml_d
#     cell = mp.Vector3(total_x, sy_c, sz_c)
#     half_tx = total_x / 2
#     x_src = -half_tx + pml_d + 0.5
#     x_mon = +half_tx - pml_d - 0.5

#     geom = [
#         mp.Block(mp.Vector3(total_x, sy_c, 2.0),
#                  center=mp.Vector3(0, 0, -(h/2 + 1.0)), material=SiO2),
#         mp.Block(mp.Vector3(total_x, wg_w, h),
#                  center=mp.Vector3(0, 0, 0), material=LNO),
#     ]
#     src = [mp.EigenModeSource(
#         mp.GaussianSource(f0, fwidth=df),
#         center=mp.Vector3(x_src, 0, 0),
#         size=mp.Vector3(0, wg_w + 1.5, h + 1.5),
#         eig_band=1, eig_parity=mp.ODD_Y + mp.EVEN_Z, direction=mp.X,
#     )]
#     sim = mp.Simulation(
#         cell_size=cell, boundary_layers=[mp.PML(pml_d)],
#         geometry=geom, sources=src, resolution=resolution,
#         default_material=mp.Medium(index=1.0),
#     )
#     flux = sim.add_flux(f0, df, 100,
#                         mp.FluxRegion(center=mp.Vector3(x_mon, 0, 0),
#                                       size=mp.Vector3(0, wg_w + 1.0, h + 1.0)))
#     sim.run(until_after_sources=mp.stop_when_fields_decayed(
#         50, mp.Ex, mp.Vector3(x_mon, 0, 0), 1e-4))
#     P_ref = mp.get_fluxes(flux)[0]
#     sim.reset_meep()
#     return P_ref



In [24]:
# ── Reference simulation (straight waveguide, for normalization) ──────────
###################### 2D #################################
def run_reference(resolution=10):
    """Straight waveguide in 2D — measures input power P_ref."""
    ref_len = 20.0
    total_x = pml_d + wg_len + ref_len + wg_len + pml_d
    sy_c = wg_w + 3.0 + 2 * pml_d

    cell = mp.Vector3(total_x, sy_c)   # 2D — no z
    half_tx = total_x / 2
    x_src = -half_tx + pml_d + 0.5
    x_mon = +half_tx - pml_d - 0.5

    geom = [
        mp.Block(
            size=mp.Vector3(total_x, wg_w),
            center=mp.Vector3(0, 0),
            material=LNO
        ),
    ]

    src = [mp.EigenModeSource(
        mp.GaussianSource(f0, fwidth=df),
        center=mp.Vector3(x_src, 0),
        size=mp.Vector3(0, wg_w + 1.5),
        eig_band=1,
        eig_parity=mp.ODD_Y,
        direction=mp.X,
    )]

    sim = mp.Simulation(
        cell_size=cell,
        dimensions=2,                        # 2D
        boundary_layers=[mp.PML(pml_d)],
        geometry=geom,
        sources=src,
        resolution=resolution,
        default_material=mp.Medium(index=n_ox),   # background = SiO2
    )

    flux = sim.add_flux(f0, df, 100,
        mp.FluxRegion(
            center=mp.Vector3(x_mon, 0),
            size=mp.Vector3(0, wg_w + 1.0)
        ))

    sim.run(until_after_sources=mp.stop_when_fields_decayed(
        50, mp.Ez,                           # Ez in 2D TE
        mp.Vector3(x_mon, 0),
        1e-4
    ))

    P_ref = mp.get_fluxes(flux)[0]
    sim.reset_meep()
    return P_ref

In [ ]:

# ── Main sweep ─────────────────────────────────────────────────────────────
print("Running reference simulation...")
P_ref = run_reference(resolution=20)
print(f"P_ref = {P_ref:.6f}")

L_sweep = np.arange(10, 56, 2)   # coarse: 10 to 54 µm in 2 µm steps
T_upper_list, T_lower_list = [], []

print(f"\n{'L (µm)':>8}  {'T_upper':>9}  {'T_lower':>9}  {'T_total':>9}  {'ratio':>7}")
for L in L_sweep:
    tu, tl = run_mmi_sim(L, resolution=20)
    tu_norm = tu / P_ref
    tl_norm = tl / P_ref
    tt = tu_norm + tl_norm
    ratio = tu_norm / tt if tt > 0 else 0
    T_upper_list.append(tu_norm)
    T_lower_list.append(tl_norm)
    print(f"{L:>8.0f}  {tu_norm:>9.4f}  {tl_norm:>9.4f}  {tt:>9.4f}  {ratio:>7.3f}")

T_upper = np.array(T_upper_list)
T_lower = np.array(T_lower_list)
T_total = T_upper + T_lower
imbalance = np.abs(T_upper - T_lower)

# Candidate points: high total T and low imbalance
valid = T_total > 0.80 * np.max(T_total)
if valid.any():
    best_idx = np.argmin(imbalance[valid])
    best_L = L_sweep[valid][best_idx]
    print(f"\nCoarse optimum: L ≈ {best_L} µm")
    print(f"At best_L: T_upper={T_upper[valid][best_idx]:.4f}, "
          f"T_lower={T_lower[valid][best_idx]:.4f}")
else:
    print("No optimum found in sweep range — extend range!")
    best_L = None


Running reference simulation...
-----------
Initializing structure...
time for choose_chunkdivision = 0.000256269 s
Working in 2D dimensions.
Computational cell is 29 x 6.9 x 0 with resolution 20
     block, center = (0,0,0)
          size (29,0.9,0)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (4.57104,4.88852,4.88852)
time for set_epsilon = 0.111615 s
-----------
MPB solved for frequency_1(1.41022,0,0) = 0.676616 after 16 iters
MPB solved for frequency_1(1.33778,0,0) = 0.645199 after 7 iters
MPB solved for frequency_1(1.33769,0,0) = 0.645161 after 3 iters
MPB solved for frequency_1(1.33769,0,0) = 0.645161 after 1 iters
field decay(t = 50.025000000000006): 1.014018322540601e-58 / 1.014018322540601e-58 = 1.0
field decay(t = 100.05000000000001): 4.491974694886881e-48 / 4.491974694886881e-48 = 1.0
on time step 5261 (time=131.525), 0.00076032 s/step
field decay(t = 150.07500000000002): 1.4195217061210088e-46 / 1.4195217061210088e-46 = 1.0
field

In [ ]:
# ── Fine sweep around best_L ────────────────────────────────────────────────
if best_L is not None:
    L_fine = np.linspace(best_L - 3, best_L + 3, 13)
    Tu_fine, Tl_fine = [], []
    for L in L_fine:
        tu, tl = run_mmi_sim(L, resolution=35)  # higher res for accuracy
        Tu_fine.append(tu / P_ref)
        Tl_fine.append(tl / P_ref)
    Tu_fine = np.array(Tu_fine)
    Tl_fine = np.array(Tl_fine)
    best_fine_idx = np.argmin(np.abs(Tu_fine - Tl_fine))
    L_opt = L_fine[best_fine_idx]
    IL_dB = -10 * np.log10(Tu_fine[best_fine_idx] + Tl_fine[best_fine_idx])
    split_pct = 100 * Tu_fine[best_fine_idx] / (Tu_fine[best_fine_idx] + Tl_fine[best_fine_idx])
    print(f"\nFine optimum: L_opt = {L_opt:.2f} µm")
    print(f"Split ratio: {split_pct:.1f}% / {100-split_pct:.1f}%")
    print(f"Insertion loss: {IL_dB:.2f} dB")
else:
    L_opt = None


In [ ]:


# ── Save ─────────────────────────────────────────────────────────────────────
scipy.io.savemat('MMI_transmission.mat', {
    'L_sweep': L_sweep,
    'T_upper': T_upper, 'T_lower': T_lower, 'T_total': T_total,
    'L_opt': L_opt if L_opt else 0,
})
print("\nSaved: MMI_transmission.mat")



In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(L_sweep, T_upper, 'b-o', label='T_out1 (upper)')
plt.plot(L_sweep, T_lower, 'r-o', label='T_out2 (lower)')
plt.plot(L_sweep, T_total, 'k--', label='T_total')
plt.axhline(0.5, color='gray', linestyle=':', label='50% line')
if L_opt: plt.axvline(L_opt, color='green', linestyle='--', label=f'L_opt={L_opt:.1f} µm')
plt.xlabel('MMI length (µm)')
plt.ylabel('Normalized transmission')
plt.title('2×2 MMI Splitting vs. Length (MEEP 3D)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('mmi_sweep.png', dpi=150)